<a href="https://colab.research.google.com/github/marcopeduto/marcopeduto/blob/main/Copie_de_Assignment_part_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### DSML investigation

You are part of the Suisse Impossible Mission Force, or SIMF for short. You need to uncover a rogue agent that is trying to steal sensitive information.

Your mission, should you choose to accept it, is to find that agent before stealing any classified information. Good luck!

# Assignment part four

#### Identifying the suspects' credit score
We received information that the rogue agent has a *good* credit score.

Our spies at SIMF have managed to collect financial information relating to our suspects as well as a training dataset.

Create a Neural Network over the training dataset `df` to identify which of the suspects have a *"Good"* `Credit_Mix`.


## Getting to know our data

* `Age`: a user's age

* `Occupation`: a user's employment field

* `Annual_Income`: a user's annual income

* `Monthly_Inh_Salary`: the calculated salary received by a given user on a monthly basis

* `Num_Bank_Accounts`: the number of bank accounts possessed by a given user

* `Num_Credit_Cards`: the number of credit cards a given user possesses

* `Interest_Rate`: The interest rate on those cards (if multiple then it's the average)

* `Num_of_Loans`: The number of loans of each user

* `Delay_from_due_date`: payment tardiness of user

* `Num_of_Delayed_Payment`: the count of delayed payments

* `Changed_Credit_Limit`: NaN

* `Num_Credit_Inquiries`: NaN

* `Credit_Mix`: The user's credit score

* `Outsting_Debt`: Outstanding debt

* `Credit_Utilization_Ratio`: the percentage of borrowed money over borrowing allowance

* `Payment_of_Min_Amount`: does the user usually pay the minimal amount (categorical)

* `Total_EMI_per_month`: Monthly repayments to be made

* `Amount_invested_monthly`: The amount put in an investment fund by the user on a monthly basis

* `Payment_Behaviour`: the user's payment behavior (categorical)

* `Monthly_Balance`: The user's end of the month balance

* `AutoLoan`: If the user has an active loan for their vehicle

* `Credit-BuilderLoan`: If the user has a loan to increase their credit score

* `DebtConsolidationLoan`, `HomeEquityLoan`, `MortgageLoan`, `NotSpecified`, `PaydayLoan`, `PersonalLoan`, `StudentLoan`: different types of loans (categorical features)


In [2]:
# Import required packages

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler

%matplotlib inline

In [3]:
df = pd.read_csv("https://raw.githubusercontent.com/michalis0/MGT-502-Data-Science-and-Machine-Learning/refs/heads/main/assignments/Part%204/data/train_classification.csv", index_col='Unnamed: 0').dropna()
df.reset_index(inplace=True, drop=True)
suspects = pd.read_csv("https://raw.githubusercontent.com/michalis0/MGT-502-Data-Science-and-Machine-Learning/refs/heads/main/assignments/Part%204/data/suspects.csv", index_col='Unnamed: 0').dropna()
suspects.rename(columns={"Payment_Behaviour": "le_Payment_Behaviour", "Payment_of_Min_Amount": "le_Payment_of_Min_Amount"}, inplace=True)

In [4]:
display(df.head())
print(df.shape)
display(suspects.head())
print(suspects.shape)

,Age,Occupation,Annual_Income,Monthly_Inh_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Monthly_Balance,AutoLoan,Credit-BuilderLoan,DebtConsolidationLoan,HomeEquityLoan,MortgageLoan,NotSpecified,PaydayLoan,PersonalLoan,StudentLoan
0,23,Scientist,19114.12,1824.843333,3,4,3,4,3,7,...,186.266702,1,1,0,1,0,0,0,1,0
1,24,Scientist,19114.12,1824.843333,3,4,3,4,3,9,...,361.444004,1,1,0,1,0,0,0,1,0
2,24,Scientist,19114.12,4182.004291,3,4,3,4,4,5,...,343.826873,1,1,0,1,0,0,0,1,0
3,28,Teacher,34847.84,3037.986667,2,4,6,1,3,3,...,303.355083,0,1,0,0,0,0,0,0,0
4,35,Engineer,143162.64,4182.004291,1,5,8,3,8,1942,...,854.226027,2,0,0,0,0,1,0,0,0


(29223, 29)


,Age,Annual_Income,Monthly_Inh_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,...,Occupation_Journalist,Occupation_Lawyer,Occupation_Manager,Occupation_Mechanic,Occupation_MediaManager,Occupation_Musician,Occupation_Scientist,Occupation_Teacher,Occupation_Writer,userID
0,23.0,19114.12,1824.843333,3.0,4.0,3.0,4.0,3.0,7.0,11.27,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,317991
1,24.0,19114.12,1824.843333,3.0,4.0,3.0,4.0,3.0,9.0,13.27,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,241892
3,24.0,19114.12,4182.004291,3.0,4.0,3.0,4.0,4.0,5.0,11.27,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,303376
5,28.0,34847.84,3037.986667,2.0,4.0,6.0,1.0,3.0,3.0,5.42,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,761992
8,35.0,143162.64,4182.004291,1.0,5.0,8.0,3.0,8.0,1942.0,7.10,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,373318


(714, 43)


In [5]:
df["Credit_Mix"].unique()

array(['Good', 'Standard', 'Bad'], dtype=object)

# 1. Preparing the data
## 1.1 Data cleaning
Consider the dataset loaded into the DataFrame `df` which is *train_classification.csv*. We aim to preprocess this data for model training.

Begin by encoding the categorical variables:
- Apply One-Hot Encoding to `Occupation`
- Apply Label Encoding to `Payment_of_Min_Amount` and `Payment_Behaviour`

*Note:*
- To clearly distinguish your newly encoded columns, for label encoding, please use `le_Payment_of_Min_Amount` and `le_Payment_Behaviour` for the label-encoded new columns.
- For the one-hot encoding, use `get_feature_names_out()` from `OneHotEncoder` to get the column names for the newly encoded columns.

In [6]:
# Your code here
onehotencoder = OneHotEncoder(sparse_output=False)
occupation_fit_transform = onehotencoder.fit_transform(df[['Occupation']])
occupation_get_feature_names_out = onehotencoder.get_feature_names_out(['Occupation'])
le1 = LabelEncoder()
le2 = LabelEncoder()
enc1 = le1.fit_transform(df['Payment_of_Min_Amount'])
enc2 = le2.fit_transform(df['Payment_Behaviour'])

 After encoding, integrate the newly encoded columns and the original `df` dataframe into a new dataframe named `df_encoded`, and remove the original columns `Occupation`, `Payment_of_Min_Amount`, and `Payment_Behaviour` to avoid redundancy.

In [7]:
# Your code here
df_encoded = df.copy()
df_encoded[occupation_get_feature_names_out] = occupation_fit_transform
df_encoded['le_Payment_of_Min_Amount'] = enc1
df_encoded['le_Payment_Behaviour'] = enc2
df_encoded = df_encoded.drop(columns=['Occupation', 'Payment_of_Min_Amount', 'Payment_Behaviour'])

Finally, display the first few rows of `df_encoded` to verify that the encodings are correctly implemented. This prepared dataframe will be used for subsequent model training.

In [8]:
# Your code here
df_encoded.head()

,Age,Annual_Income,Monthly_Inh_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,...,Occupation_Lawyer,Occupation_Manager,Occupation_Mechanic,Occupation_MediaManager,Occupation_Musician,Occupation_Scientist,Occupation_Teacher,Occupation_Writer,le_Payment_of_Min_Amount,le_Payment_Behaviour
0,23,19114.12,1824.843333,3,4,3,4,3,7,11.27,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1,6
1,24,19114.12,1824.843333,3,4,3,4,3,9,13.27,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1,2
2,24,19114.12,4182.004291,3,4,3,4,4,5,11.27,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1,2
3,28,34847.84,3037.986667,2,4,6,1,3,3,5.42,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,4
4,35,143162.64,4182.004291,1,5,8,3,8,1942,7.10,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,5


## 1.2 Dataset splitting and rescaling

To effectively train and validate our model, it is crucial to properly prepare and partition the data. Follow these steps to preprocess and split the dataframe `df_encoded` into training and test subsets, ensuring that our model can generalize well to new data:

- Set `X` as all columns except `Credit_Mix` and `y` as the dependent feature `Credit_Mix`.
- Manually apply label encoding to `y` using the function `.map` and encode with the following: Good=2, Standard=1, Bad=0.
- Use a `random_state` of 42 to split `X` and the encoded `y` into training (80%) and test sets (20%).
- Normalize `X` using `MinMaxScaler()`.

In [9]:
# Your code here
X = df_encoded.drop(columns=['Credit_Mix'])
y = df_encoded['Credit_Mix'].map({'Good': 2, 'Standard': 1, 'Bad': 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Convert the features to torch tensors of type `torch.float` and the labels (dependent variables) to torch tensors of type `torch.long`.

In [10]:
#Your code here
X_train_tensor = torch.tensor(X_train, dtype=torch.float)
X_test_tensor = torch.tensor(X_test, dtype=torch.float)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

# 2. Model defining and hyperparameter tuning
## 2.1 Define a neural network and instantiate it
Define your neural network model as a class in PyTorch, extending from `nn.Module`. In the `__init__` method, initialize a linear layer using `nn.Linear()` with specified input and output sizes, and set up `nn.ReLU()` for activation. Implement the forward method to describe how data passes through this layer during the network's forward computation.

Your neural network should have:
* 1 hidden layer
* ReLU as the activation function

In [11]:
# Your code here

Set `D_in` to the number of features in `X_train` and `D_out` to the number of target variables in `y_train`, then print these dimensions to verify their values.

In [12]:
# Your code here
D_in = X_train_tensor.shape[1]
D_out = len(torch.unique(y_train_tensor))
print(D_in)
print(D_out)

42
3



Initialize the `Net` model with the specified input size `D_in`, 150 hidden units, and output size `D_out`.

In [13]:
# Your code here
class Net(nn.Module):
    def __init__(self, D_in, H, D_out):
        super(Net, self).__init__()
        self.linear1 = nn.Linear(D_in, H)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(H, D_out)
    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out
model = Net(D_in=D_in, H=150, D_out=D_out)

Let's calculate now how many parameters we have in the model.

In [14]:
# Your code here
parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(parameters)

6903


**Q1. How many parameters does your model have ?**

*Note: Enter an integer (e.g. 355)*

In [15]:
parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(parameters)

6903


## 2.2 Finding the best model

We will use the following criterion and optimizer:
* `criterion` : [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
* `optimizer` : [Stochastic Gradient Descent (SGD)](https://en.wikipedia.org/wiki/Stochastic_gradient_descent)

Since we don't know the optimal number of epochs and learning rates yet, we will use a loop to test different hyperparameter combinations. This will help us identify which pair yields the best results.

Specifically, we will test the following values:
* `Epochs`: **150**, **250**, **500**, and **1000** epochs.
* `Learning Rate`: **0.00005**, **0.001**, **1**, and **10**.

For each combination, you will need to define and train a new model, then assess its performance by measuring its accuracy on the test set.

*Note: Run the code `torch.manual_seed(42)` to ensure consistency across all experiments.*

In [16]:
torch.manual_seed(42)   # Set the seed for reproducibility

Begin by defining the ranges for the hyperparameters you want to explore as specified above.

In [17]:
# Your code here
epochs = [150, 250, 500, 1000]
learningrates = [0.00005, 0.001, 1, 10]

Next, create a loop that iterates over the list of learning rates, with an inner loop iterating over the list of number of epochs. Inside the inner loop, initialize (define) the model, and also define the optimizer and the loss function (criterion). Then train the model as demonstrated in the lab, and after the training, evaluate its performance to obtain the test accuracy for each model. Ensure that you display all the test accuracies, as you will need them to answer the upcoming questions.

Here’s a reminder of the criterion and optimizer you should use:
- **Criterion**: [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- **Optimizer**: [Stochastic Gradient Descent (SGD)](https://en.wikipedia.org/wiki/Stochastic_gradient_descent)

**Hint**: Make sure to define the criterion and optimizer at every iteration (inside the inner loop), otherwise the model might retain previous training states and produce biased results.

(Optional) You can enhance the code by storing the best model along with its best_accuracy and best_params. This way, you'll have an automatic evaluation at the end to identify the best-performing model.


In [18]:
# Your code here

# énorme bug ici, car j'ai eu beaucoup de peine à run à chaque fois que je modifiais qlqch

best_accuracy = 0.0
best_hyperparams = {'lr': None, 'epochs': None}

for lr in learningrates:
    for ep in epochs :
        torch.manual_seed(42)
        model = Net(D_in=D_in, H=150, D_out=D_out)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=lr)

        for epoch in range(ep):
            model.train()
            optimizer.zero_grad()
            outputs = model(X_train_tensor)
            loss = criterion(outputs, y_train_tensor)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test_tensor)
            _, predicted = torch.max(test_outputs, 1)
            correct = (predicted == y_test_tensor).sum().item()
            accuracy = correct / y_test_tensor.size(0)

        print(f"learning rate : {lr} | epochs : {ep} | Accuracy: {accuracy}")

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_hyperparams = {'lr': lr, 'epochs': ep}

print("\n--- MEILLEUR MODÈLE ---")
print(f"Meilleure accuracy : {best_accuracy:.4f}")
print(f"Meilleurs hyperparamètres : {best_hyperparams}")

learning rate : 5e-05 | epochs : 150 | Accuracy: 0.30966638152266895
learning rate : 5e-05 | epochs : 250 | Accuracy: 0.3142857142857143
learning rate : 5e-05 | epochs : 500 | Accuracy: 0.33088109495295126
learning rate : 5e-05 | epochs : 1000 | Accuracy: 0.3627031650983747
learning rate : 0.001 | epochs : 150 | Accuracy: 0.42087254063301965
learning rate : 0.001 | epochs : 250 | Accuracy: 0.45406330196749356
learning rate : 0.001 | epochs : 500 | Accuracy: 0.4544054747647562
learning rate : 0.001 | epochs : 1000 | Accuracy: 0.4545765611633875
learning rate : 1 | epochs : 150 | Accuracy: 0.8135158254918734
learning rate : 1 | epochs : 250 | Accuracy: 0.8306244653550042
learning rate : 1 | epochs : 500 | Accuracy: 0.8407185628742515
learning rate : 1 | epochs : 1000 | Accuracy: 0.8449957228400342
learning rate : 10 | epochs : 150 | Accuracy: 0.4545765611633875
learning rate : 10 | epochs : 250 | Accuracy: 0.4545765611633875
learning rate : 10 | epochs : 500 | Accuracy: 0.454576561163387

**Questions**

**Q2. What is the best test accuracy we can achieve when training the model using different combinations of learning rates (0.00005, 0.001, 1, 10) and epochs (150, 250, 500, 1000)?**

*Note: Enter a float rounded to 2 decimal points (e.g., 0.78, not 78).*

**Q3. When using 1000 epochs, which learning rate results in the highest test accuracy?**

*Note: Select among the following answers*


**Q4. Is BCELoss a suitable alternative to CrossEntropyLoss for our dataset?**

*Hint: Consider the unique values in the Credit_Mix output variable when answering.*

# 3. Predicting on the Suspects Dataset

Now it's time to use your trained model to make predictions on the suspects dataset!

Please retrain on the full dataset (i.e. train again on the entire `df_encoded` dataframe).

*Note: Remember to initialize another `MinMaxScaler` to scale the training and test data as they are now different compared to the previous steps*

Use the following parameters for the model:
- **Hidden layer**: 1 hidden layer with 150 neurons
- **Output layer**: 3 neurons for classification
- **Optimizer**: [Stochastic Gradient Descent (SGD)](https://en.wikipedia.org/wiki/Stochastic_gradient_descent)
- **Criterion**: [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- **Iterations**: 1000
- **Learning rate**: 1.0

Ensure consistency by setting the manual seed with `torch.manual_seed(42)` before training.

In [19]:
# Your code here
torch.manual_seed(42)
model_full = Net(D_in=D_in, H=150, D_out=D_out)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_full.parameters(), lr=1.0)


Now, train your model on the training dataset just as you did in section 2.2.1, but you don't have to loop over all the hyper-parameters. Ensure that you use the correct number of epochs and the correct value of learning rate specified earlier.

In [20]:
# Your code here
X_full = df_encoded.drop(columns=['Credit_Mix'])
y_full = df_encoded['Credit_Mix'].map({'Good': 2, 'Standard': 1, 'Bad': 0})
scaler_full = MinMaxScaler()
X_full_scaled = scaler_full.fit_transform(X_full)
X_full_tensor = torch.tensor(X_full_scaled, dtype=torch.float)
y_full_tensor = torch.tensor(y_full.values, dtype=torch.long)

for epoch in range(1000):
    model_full.train()
    optimizer.zero_grad()
    outputs = model_full(X_full_tensor)
    loss = criterion(outputs, y_full_tensor)
    loss.backward()
    optimizer.step()

Before making predictions, confirm that the feature column names in the `suspects` dataset match those expected by the model, particularly ensuring the X features correspond accurately.

In [21]:
# Your code here
suspects_features = suspects.copy()
for col in X_full.columns:
    if col not in suspects_features.columns:
        suspects_features[col] = 0
suspects_features = suspects_features[X_full.columns]

Then scale your dataset and convert it into a torch tensor of dtype float, similar to the preprocessing done for the training set in section 1.2.

In [22]:
# Your code here
suspects_scaled = scaler_full.transform(suspects_features)
suspects_tensor = torch.tensor(suspects_scaled, dtype=torch.float)

Make predictions using the trained model and assign the predicted credit score to each user. Ensure that the following encoding is used for the predicted categories:
* `0` corresponds to bad credit score,
* `1` corresponds to standard credit score,
* `2` corresponds to good credit score.

Use the predictions to add a new column `credit_score` in the `suspects` dataframe that maps the predicted numerical values to the respective credit score categories.

In [23]:
# Your code here
model_full.eval()
with torch.no_grad():
    preds = model_full(suspects_tensor)
    _, predicted_classes = torch.max(preds, 1)
suspects['credit_score'] = predicted_classes.numpy()



As mentioned earlier, we believe the suspect had a "Good" credit score. Review the predictions made by the model, then display how many suspects were categorized under each credit score, and extract the `userID`s of those with a "Good" credit score.

In [24]:
# Your code here
print(suspects['credit_score'].value_counts())
good_suspects = suspects[suspects['credit_score'] == 2]
display(good_suspects)

credit_score
2    267
1    260
0    187
Name: count, dtype: int64


,Age,Annual_Income,Monthly_Inh_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,...,Occupation_Lawyer,Occupation_Manager,Occupation_Mechanic,Occupation_MediaManager,Occupation_Musician,Occupation_Scientist,Occupation_Teacher,Occupation_Writer,userID,credit_score
0,23.0,19114.12,1824.843333,3.0,4.0,3.0,4.0,3.0,7.0,11.27,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,317991,2
1,24.0,19114.12,1824.843333,3.0,4.0,3.0,4.0,3.0,9.0,13.27,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,241892,2
3,24.0,19114.12,4182.004291,3.0,4.0,3.0,4.0,4.0,5.0,11.27,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,303376,2
5,28.0,34847.84,3037.986667,2.0,4.0,6.0,1.0,3.0,3.0,5.42,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,761992,2
8,35.0,143162.64,4182.004291,1.0,5.0,8.0,3.0,8.0,1942.0,7.10,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,373318,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1201,47.0,94511.79,7717.982500,5.0,1434.0,9.0,0.0,1.0,11.0,10.29,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,868229,2
1203,48.0,94511.79,7717.982500,5.0,3.0,9.0,0.0,1.0,9.0,7.29,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,789469,2
1217,31.0,73434.92,5849.576667,1.0,5.0,5.0,4.0,6.0,3.0,11.93,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,866017,2
1218,31.0,73434.92,5849.576667,1.0,5.0,5.0,4.0,6.0,3.0,11.93,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,887305,2


**Q5. Which of the following suspects have a "good" credit mix according to your model's predictions?**


In [32]:
choix = [167822, 761992, 519735, 862880, 409055]
if 'userID' in suspects.columns:
    matched = suspects[suspects['userID'].isin(choix) & (suspects['credit_score'] == 2)]
    print()
    display(matched[['userID', 'credit_score']])
else:
    print(suspects.columns)

,userID,credit_score
5,761992,2
123,167822,2
1177,409055,2


## Your investigation is progressing effectively, and the list of suspects is narrowing down.

**Don't forget to answer the quiz and submit your code on Moodle before the end of the deadline.**